[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/poncelet/poncelet.ipynb)

# Poncelet's Porism

This example traces polygonal paths inscribed in an outer conic and circumscribed about an inner conic to demonstrate Poncelet's porism. Starting from a point on the outer ellipsoid, a path follows a tangent of the inner conic to its second intersection with the ellipsoid, turns onto the other tangent through that vertex, and repeats from vertex to vertex. Poncelet's porism states that closure depends only on the two conics, so a path that closes after a given number of sides from one starting point closes after the same number of sides from every starting point on the outer conic, while a pair that misses from one starting point misses from all of them.

In projective geometric algebra, points are modelled as intrinsic antivectors and lines as point joins `vertex & heading`, while polar planes are extrinsic vectors and poles are plane meets. An outer point conic is modelled as a polarity map sending each point to its polar plane, vanishing on the curve where `outer(vertex) & vertex == 0`, and an inner line conic is modelled dually as an envelope map sending each line to its pole, vanishing on tangents where `inner(side) & side == 0`. Two poles `first_corner` and `opposite_corner` form a degenerate envelope through the symmetric dyad `first_corner * (opposite_corner & Line) + opposite_corner * (first_corner & Line)`, so five tangent lines fix the unique envelope touching them as a linear combination of two quadrilateral pole pairs. Once one root on a line or in a vertex pencil is known, Vieta's relation gives the next crossing `vertex * (outer(heading) & heading) - heading * (2 * (outer(vertex) & heading))` and the outgoing tangent `heading * (inner(spoke_line) & spoke_line) - spoke * (2 * (inner(spoke_line) & side))` rationally without further square roots.

In homogeneous projective matrix coordinates, a point $\mathbf{x} = [x, y, 1]^\top$ lies on a point conic with $3 \times 3$ symmetric matrix $Q$ when $\mathbf{x}^\top Q \mathbf{x} = 0$, while a line $\boldsymbol{\ell} = \mathbf{x}_1 \times \mathbf{x}_2$ touches a dual conic $C$ when $\boldsymbol{\ell}^\top C \boldsymbol{\ell} = 0$. Fitting $C$ to five tangent lines $\boldsymbol{\ell}_1, \dots, \boldsymbol{\ell}_5$ combines two symmetrized cross-product outer products $(\boldsymbol{\ell}_1 \times \boldsymbol{\ell}_2)(\boldsymbol{\ell}_3 \times \boldsymbol{\ell}_4)^\top + (\boldsymbol{\ell}_3 \times \boldsymbol{\ell}_4)(\boldsymbol{\ell}_1 \times \boldsymbol{\ell}_2)^\top$ and $(\boldsymbol{\ell}_1 \times \boldsymbol{\ell}_3)(\boldsymbol{\ell}_2 \times \boldsymbol{\ell}_4)^\top + (\boldsymbol{\ell}_2 \times \boldsymbol{\ell}_4)(\boldsymbol{\ell}_1 \times \boldsymbol{\ell}_3)^\top$, and stepping along $\mathbf{d} = [d_x, d_y, 0]^\top$ evaluates $(\mathbf{d}^\top Q \mathbf{d})\mathbf{x} - 2(\mathbf{x}^\top Q \mathbf{d})\mathbf{d}$.

In [ ]:
# Install numga and its example dependencies when running in Google Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_states
from examples.math.poncelet import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# Points are intrinsic positions, while polar planes are extrinsic linear forms on points.
Point = ga.gatype.antivector()                                                 # Antivector-
Plane = ga.gatype.vector()                                                     # Vector+
# Joining two points forms an intrinsic line, while intersecting two planes forms an extrinsic pole.
Line = ga.gatype(Point & Point)                                                # Antibivector-
Pole = ga.gatype(Plane ^ Plane)                                                # Bivector+
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
Envelope = ga.gatype((Pole, Line))                                             # Pole+ <- Line-
Motor = ga.gatype.rotor()                                                      # Rotor+
# The two sign choices select the two quadratic roots in `tangents`.
root_signs = np.array([1, -1])                                                 # [2]

## Point quadrics and line envelopes

An origin-centred ellipsoid sums the dyads of its principal planes weighted by the inverse squares of its semi-axes and subtracts the horizon plane dyad, after which a rigid motor carries those planes to the ellipsoid's centre and tilt. Dually, five tangent lines determine a unique envelope because the opposite corner pairs of the first four lines span a one-parameter pencil of degenerate two-pole envelopes and vanishing on the fifth line selects one member. Comparing the five sides of an inscribed pentagon against the sides of a slightly scaled copy produces one inner quadric that admits closed five-sided Poncelet polygons and a neighbouring inner quadric that does not.

In [ ]:
# Ellipsoid centre, principal semi-axes, and tilt angle from `x`.
centre = np.array([0.4, -0.2])                                                 # [2]
semi_axes = np.array([2.0, 1.2])                                               # [2]
tilt = 0.5
# Parameter angles of the five pentagon vertices, cyclic next-vertex indices, and radial scales of the two pentagons.
corner_angles = np.array([0.1, 0.8, 1.9, 3.0, 5.2])                            # [5]
next_corners = np.array([1, 2, 3, 4, 0])                                       # [5]
pentagon_scales = np.array([1.0, 0.97])                                        # [2]

In [ ]:
def ellipsoid(semi_axes: np.ndarray, motor: Motor) -> Quadric:
    """Principal planes carried from the origin by `motor` and weighted by `semi_axes` (`[2]`)
    define the ellipsoid's polarity map (`[] Plane+ <- Point-`)."""
    axes = motor >> stack([mv.x, mv.y])                                        # [2] Plane+
    axial_part = (axes * (axes & Point) / semi_axes**2).sum(axis=-1)           # [] Plane+ <- Point-
    return axial_part - mv.w * (mv.w & Point)                                  # [] Plane+ <- Point-


def envelope(sides: Line) -> Envelope:
    """Five tangent lines (`[..., 5] Line-`) determine the unique envelope (`[...] Pole+ <- Line-`)
    touching all five."""
    first, second, third, fourth, fifth = (sides[..., index] for index in range(5))
    # Intersecting the first four lines in two pairings gives opposite vertices of their quadrilateral.
    first_corner, opposite_corner = first ^ second, third ^ fourth             # [...] Pole+, Pole+
    first_cross, opposite_cross = first ^ third, second ^ fourth               # [...] Pole+, Pole+
    # Each symmetric pole dyad is a degenerate envelope vanishing on every line through either pole.
    corner_pair = (first_corner * (opposite_corner & Line) + opposite_corner * (first_corner & Line)) / 2   # [...] Pole+ <- Line-
    cross_pair = (first_cross * (opposite_cross & Line) + opposite_cross * (first_cross & Line)) / 2        # [...] Pole+ <- Line-
    return corner_pair * (cross_pair(fifth) & fifth) - cross_pair * (corner_pair(fifth) & fifth)            # [...] Pole+ <- Line-


# Rigid motor turning the ellipsoid by `tilt` and translating its origin to `centre`.
centre_shift = ((mv.xw * centre[0] + mv.yw * centre[1]) * 0.5).exp()           # [] Motor+
tilt_turn = (mv.xy * (-tilt / 2)).exp()                                        # [] Motor+
placement = centre_shift * tilt_turn                                           # [] Motor+

outer = ellipsoid(semi_axes, placement)                                        # [] Plane+ <- Point-
unit_corners = np.stack([np.cos(corner_angles), np.sin(corner_angles)], axis=-1)   # [5, 2]
corners = placement >> mv.point(pentagon_scales[:, None, None] * semi_axes * unit_corners)   # [2, 5] Point-
inner = envelope(corners & corners[:, next_corners])                           # [2] Pole+ <- Line-

In [ ]:
print("outer ellipsoid polarity:\n", outer.coefficient_table())
print("inscribed-pentagon envelope:\n", inner[0].coefficient_table())

## Rational crossings and tangent turns

Joining a vertex on the outer ellipsoid with a direction at infinity parameterizes the line along that direction, on which the outer polarity vanishes at the vertex itself and at one other point given rationally by the linear and quadratic pairings. Dually, joining the new vertex with a fixed interior point gives a spoke line that never touches the inner envelope, so the incoming side and the spoke span the pencil of lines through the vertex and yield the outgoing tangent by the same rational root relation. Only the initial tangent direction from a starting point requires a square root because neither root of the vertex pencil is known in advance.

In [ ]:
# Side count of each polygon and parameter angle of the static sample start on the ellipsoid.
n_sides = 5
sample_angle = 0.05 + np.pi / 6

In [ ]:
def tangents(inner: Envelope, points: Point) -> Point:
    """The line pencil through each exterior point in `points` (`[...] Point-`) has two tangent
    directions (`[..., 2] Point-`) at infinity to `inner` (`[...] Pole+ <- Line-`)."""
    # Joining each point with the coordinate directions at infinity spans the pencil of lines through it.
    x_direction, y_direction = mv.x.dual(), mv.y.dual()                        # [] Point-, Point-
    x_lines, y_lines = points & x_direction, points & y_direction              # [...] Line-, Line-
    xx_term = inner(x_lines) & x_lines                                         # [...] Scalar+
    xy_term = inner(y_lines) & x_lines                                         # [...] Scalar+
    yy_term = inner(y_lines) & y_lines                                         # [...] Scalar+
    # Outside `inner` the discriminant is positive and splits into the two tangent directions; on `inner` it vanishes.
    discriminant_root = (xy_term * xy_term - xx_term * yy_term).square_root()  # [...] Scalar+
    x_weights = discriminant_root[..., None] * root_signs - xy_term[..., None] # [..., 2] Scalar+
    return x_direction * x_weights + y_direction * xx_term[..., None]          # [..., 2] Point-


def path(outer: Quadric, inner: Envelope, start: Point, heading: Point, inside: Point) -> Iterator[Point]:
    """Yields successive vertices (`[...] Point-`) on `outer` by stepping along tangents of `inner`,
    using the interior point `inside` to select the outgoing tangent rationally at each vertex."""
    vertex = start.broadcast_to(heading.shape)                                 # [...] Point-
    while True:
        yield vertex
        # Along `vertex + heading * t`, `outer` vanishes at `t = 0` and at the rational second root below.
        crossing = vertex * (outer(heading) & heading) - heading * (2 * (outer(vertex) & heading))   # [...] Point-
        vertex = crossing.normalized()                                         # [...] Point-
        # Because `inside` lies strictly inside `inner`, `spoke_line` never touches `inner` and spans the pencil with `side`.
        spoke = vertex - inside                                                # [...] Point-
        side, spoke_line = vertex & heading, vertex & spoke                    # [...] Line-, Line-
        # Along `heading + spoke * t`, `inner` vanishes on the incoming `side` (`t = 0`) and on the outgoing tangent.
        outgoing = heading * (inner(spoke_line) & spoke_line) - spoke * (2 * (inner(spoke_line) & side))   # [...] Point-
        heading = outgoing / outgoing.antinorm()                               # [...] Point-


sample_direction = np.array([np.cos(sample_angle), np.sin(sample_angle)])      # [2]
sample_start = placement >> mv.point(semi_axes * sample_direction)             # [] Point-
sample_heading = tangents(inner, sample_start)[:, 0]                           # [2] Point-
# The ellipsoid centre lies strictly inside both inner envelopes, so no tangent of either passes through it.
inside = mv.point(centre)                                                      # [] Point-
sample_vertices = stack(list(islice(path(outer, inner, sample_start, sample_heading, inside), n_sides + 1)))   # [n_sides + 1, 2] Point-
save_figure(render.paths_figure(outer, inner, sample_vertices), "poncelet_pentagons");

The grey curve marks the outer ellipsoid, and each blue curve marks an inner envelope where the two tangent directions through a point coincide. Each red path traces five tangent sides from the orange starting vertex to the ringed fifth vertex, closing onto the start around the inscribed pentagon's envelope on the left and missing around the scaled pentagon's envelope on the right.

## Paths from every starting point on the ellipsoid

Sweeping the starting point all the way around the outer ellipsoid tests Poncelet's porism across the full family of inscribed paths. Around the first inner envelope every five-sided path returns to its orange starting vertex regardless of how bunched or stretched the corners become, whereas around the second inner envelope the fifth vertex stays separated from the start at every angle.

In [ ]:
# Starting angles around a full turn of the ellipsoid, frame delay in milliseconds, and GIF palette size.
n_frames = 180
start_offset = 0.05
start_angles = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False) + start_offset   # [n_frames]
frame_duration_ms = 40
n_colours = 32

In [ ]:
def polygon_paths(outer: Quadric, inner: Envelope, starts: Point, inside: Point,
                  n_sides: int) -> Iterator[Point]:
    """Yields the `[n_sides + 1, ...]` polygon vertices (`Point-`) around `inner` from each
    starting point in `starts`."""
    for start in starts:
        heading = tangents(inner, start)[..., 0]                               # [...] Point-
        yield stack(list(islice(path(outer, inner, start, heading, inside), n_sides + 1)))   # [n_sides + 1, ...] Point-


unit_starts = np.stack([np.cos(start_angles), np.sin(start_angles)], axis=-1)  # [n_frames, 2]
starts = placement >> mv.point(semi_axes * unit_starts)                        # [n_frames] Point-
circuit_states = show_states(
    polygon_paths(outer, inner, starts, inside, n_sides),
    render.path_frames, outer, inner,
    name="poncelet_turn", duration_ms=frame_duration_ms, n_frames=n_frames, colors=n_colours,
)

In [ ]:
# Subtracting two unit-weight points leaves a pure displacement whose Euclidean length is its antinorm.
vertices = stack(circuit_states)                                               # [n_frames, n_sides + 1, 2] Point-
gaps = (vertices[:, n_sides] - vertices[:, 0]).antinorm()                      # [n_frames, 2] Scalar+
print("smallest miss distance around the scaled-pentagon envelope:", gaps[:, 1].to_array().min())

In [ ]:
# checks
# Every vertex lies on the outer ellipsoid, every side touches its inner envelope, and five-sided paths
# close from every starting point around the first inner envelope while missing around the second.
assert ((outer(sample_vertices) & sample_vertices).abs() < 1e-10).all()
assert ((outer(vertices) & vertices).abs() < 1e-10).all()
sides = (vertices[:, :-1] & vertices[:, 1:]).normalized()                      # [n_frames, n_sides, 2] Line-
horizon = mv.x.dual() & mv.y.dual()                                            # [] Line-
touch = (inner(sides) & sides) / (inner(horizon) & horizon)                    # [n_frames, n_sides, 2] Scalar+
assert (touch.abs() < 1e-10).all()
assert (gaps[:, 0] < 1e-9).all()
assert (gaps[:, 1] > 1e-2).all()